# 🔄 quantum2json
**Conversión inversa: `QuantumCircuit.data` → diccionario json2quantum.**

Complemento de [json2quantum](https://github.com/lmbibbo/json2quantum): recorre `circuit.data` y reconstruye
la estructura JSON original, lista para guardar en disco o verificar round-trips.

### Mapa de compuertas soportadas
| Nombre interno Qiskit | Nombre en JSON |
|---|---|
| `x` | `X` |
| `h` | `H` |
| `z` | `Z` |
| `s` | `S` |
| `t` | `T` |
| `rz` | `Rz` |
| `ry` | `Ry` |
| `cx` | `CNOT` |

> **Nota**: `measure`, `barrier`, `reset` y `delay` se omiten automáticamente.

## 📦 1. Instalación de dependencias

In [1]:
# Instalar Qiskit si no está disponible (necesario en Colab)
try:
    import qiskit
    print(f"✅ Qiskit {qiskit.__version__} ya instalado")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "qiskit", "qiskit-aer", "pylatexenc", "-q"])
    print("✅ Qiskit instalado correctamente")

✅ Qiskit 2.4.1 ya instalado


## 📚 2. Imports

In [2]:
from __future__ import annotations

import json
from pathlib import Path

from qiskit import QuantumCircuit

print("✅ Imports OK")

✅ Imports OK


## 🗺️ 3. Mapa inverso Qiskit → JSON

In [3]:
# Mapa inverso: nombre interno Qiskit → nombre del formato json2quantum
QISKIT_TO_JSON: dict[str, str] = {
    "x":  "X",
    "h":  "H",
    "z":  "Z",
    "s":  "S",
    "t":  "T",
    "rz": "Rz",
    "ry": "Ry",
    "cx": "CNOT",
}

# Instrucciones que se ignoran (no son compuertas lógicas)
_SKIP_OPS = {"measure", "barrier", "reset", "delay"}

print("✅ Mapa definido")

✅ Mapa definido


## 🔧 4. Función `circuit_to_json`

### Lógica interna
| Paso | Acción |
|---|---|
| 1 | Construir `qubit_index`: mapa `Qubit → índice entero` |
| 2 | Iterar `circuit.data` → lista de `CircuitInstruction` |
| 3 | Saltar ops en `_SKIP_OPS` |
| 4 | Traducir `op.name` con `QISKIT_TO_JSON` |
| 5 | Separar qubits: `indices[:-1]` = controles, `indices[-1]` = target |
| 6 | Añadir `parameter` solo si `op.params` no está vacío |

In [4]:
def circuit_to_json(
    circuit: QuantumCircuit,
    level: int | str = "?",
) -> dict:
    """
    Convierte un QuantumCircuit de Qiskit al formato dict json2quantum.

    Parámetros
    ----------
    circuit : QuantumCircuit
        Circuito origen.  Puede tener o no mediciones finales;
        éstas se omiten automáticamente.
    level : int | str
        Valor para el campo ``level`` del JSON resultante.

    Retorna
    -------
    dict
        Diccionario compatible con parse_json_to_circuit.
    """
    n_qubits = circuit.num_qubits

    # Índice global de cada qubit dentro del circuito
    qubit_index: dict = {qubit: idx for idx, qubit in enumerate(circuit.qubits)}

    operations: list[dict] = []
    basis_set: set[str] = set()

    for instruction in circuit.data:
        op     = instruction.operation
        qubits = instruction.qubits

        # Saltar mediciones y otras instrucciones no cuánticas
        if op.name in _SKIP_OPS:
            continue

        json_name = QISKIT_TO_JSON.get(op.name)
        if json_name is None:
            raise ValueError(
                f"Compuerta '{op.name}' no tiene equivalente en el formato JSON.\n"
                f"  Disponibles: {list(QISKIT_TO_JSON.keys())}"
            )

        # Para compuertas de 2+ qubits:
        #   los primeros qubits son controles, el último es el target.
        indices      = [qubit_index[q] for q in qubits]
        target       = indices[-1]
        control_list = indices[:-1]   # lista vacía para compuertas de 1 qubit

        gate_def: dict = {
            "name":     json_name,
            "label":    op.label if op.label else json_name,
            "target":   target,
            "controls": control_list,
        }

        # Añadir parámetro sólo si la compuerta lo requiere
        if op.params:
            gate_def["parameter"] = round(float(op.params[0]), 10)

        operations.append({"gate": gate_def})
        basis_set.add(json_name)

    return {
        "level":      level,
        "qubits":     n_qubits,
        "basis":      sorted(basis_set),
        "operations": operations,
    }


print("✅ circuit_to_json definido")

✅ circuit_to_json definido


## 💾 5. Función `circuit_to_json_file`
Wrapper que guarda el JSON directamente a disco.

In [5]:
def circuit_to_json_file(
    circuit: QuantumCircuit,
    path: str | Path,
    level: int | str = "?",
    indent: int = 2,
) -> Path:
    """
    Wrapper que guarda el resultado de circuit_to_json directamente a disco.

    Retorna
    -------
    Path
        Ruta del archivo creado.
    """
    output = Path(path)
    data   = circuit_to_json(circuit, level=level)
    output.write_text(json.dumps(data, indent=indent), encoding="utf-8")
    return output


print("✅ circuit_to_json_file definido")

✅ circuit_to_json_file definido


---
## 🧪 6. Demos

### 6a — Bell State `|Φ+⟩`

In [6]:
# Construir el circuito directamente en Qiskit
qc_bell = QuantumCircuit(2, name="bell")
qc_bell.h(0)
qc_bell.cx(0, 1)
qc_bell.measure_all()

# Convertir a JSON
bell_json = circuit_to_json(qc_bell, level=99)

print("🔔 Bell State — JSON reconstruido desde circuit.data:")
print(json.dumps(bell_json, indent=2))

🔔 Bell State — JSON reconstruido desde circuit.data:
{
  "level": 99,
  "qubits": 2,
  "basis": [
    "CNOT",
    "H"
  ],
  "operations": [
    {
      "gate": {
        "name": "H",
        "label": "H",
        "target": 0,
        "controls": []
      }
    },
    {
      "gate": {
        "name": "CNOT",
        "label": "CNOT",
        "target": 1,
        "controls": [
          0
        ]
      }
    }
  ]
}


### 6b — Circuito con parámetro (Rz)

In [7]:
import math

qc_rz = QuantumCircuit(4, name="nivel6")
qc_rz.rz(math.pi / 4, 2)   # Rz(π/4) sobre q[2]
qc_rz.cx(0, 2)              # CNOT control=q[0] target=q[2]

rz_json = circuit_to_json(qc_rz, level=6)

print("🔁 Circuito nivel 6 — JSON reconstruido:")
print(json.dumps(rz_json, indent=2))

🔁 Circuito nivel 6 — JSON reconstruido:
{
  "level": 6,
  "qubits": 4,
  "basis": [
    "CNOT",
    "Rz"
  ],
  "operations": [
    {
      "gate": {
        "name": "Rz",
        "label": "Rz",
        "target": 2,
        "controls": [],
        "parameter": 0.7853981634
      }
    },
    {
      "gate": {
        "name": "CNOT",
        "label": "CNOT",
        "target": 2,
        "controls": [
          0
        ]
      }
    }
  ]
}


### 6c — Guardar a disco

In [8]:
out = circuit_to_json_file(qc_bell, "bell_recovered.json", level=99)
print(f"💾 Guardado en: {out.resolve()}")
print(out.read_text(encoding='utf-8'))

💾 Guardado en: C:\Users\palob\Program\Quantum\json2quantum\bell_recovered.json
{
  "level": 99,
  "qubits": 2,
  "basis": [
    "CNOT",
    "H"
  ],
  "operations": [
    {
      "gate": {
        "name": "H",
        "label": "H",
        "target": 0,
        "controls": []
      }
    },
    {
      "gate": {
        "name": "CNOT",
        "label": "CNOT",
        "target": 1,
        "controls": [
          0
        ]
      }
    }
  ]
}


---
## ✅ 7. Round-trip con json2quantum

Verifica que `circuit_to_json` → `parse_json_to_circuit` reproduce un circuito equivalente.

> Pega aquí el parser de `json2quantum.ipynb` (celda `parser-cell`) o importa el módulo si lo tienes disponible.

In [ ]:
# ── Parser copiado de json2quantum.ipynb ────────────────────────────────────
from qiskit import QuantumRegister, ClassicalRegister
from qiskit.circuit.library import (
    RZGate, RYGate, XGate, CXGate, HGate, ZGate, SGate, TGate,
)

GATE_MAP: dict = {
    "X":    lambda p: XGate(),
    "H":    lambda p: HGate(),
    "Z":    lambda p: ZGate(),
    "S":    lambda p: SGate(),
    "T":    lambda p: TGate(),
    "Rz":   lambda p: RZGate(p),
    "RZ":   lambda p: RZGate(p),
    "Ry":   lambda p: RYGate(p),
    "RY":   lambda p: RYGate(p),
    "CNOT": lambda p: CXGate(),
    "CX":   lambda p: CXGate(),
}


def _resolve_control(c) -> int:
    if isinstance(c, int):
        return c
    if isinstance(c, dict):
        return c["qubit"]
    raise ValueError(f"Formato de control no reconocido: {c!r}")


def parse_json_to_circuit(json_data: dict) -> QuantumCircuit:
    n_qubits   = json_data["qubits"]
    operations = json_data["operations"]
    level      = json_data.get("level", "?")

    qr = QuantumRegister(n_qubits, name="q")
    cr = ClassicalRegister(n_qubits, name="c")
    qc = QuantumCircuit(qr, cr, name=f"nivel{level}")

    for op in operations:
        gate_def  = op["gate"]
        name      = gate_def["name"]
        parameter = gate_def.get("parameter", None)
        controls  = gate_def.get("controls", [])
        target    = gate_def["target"]

        gate = GATE_MAP[name](parameter).to_mutable()
        gate.label = gate_def.get("label", name)

        control_indices = [_resolve_control(c) for c in controls]
        all_qubits = [qr[c] for c in control_indices] + [qr[target]]
        qc.append(gate, all_qubits)

    #sqc.measure(qr, cr)
    return qc


print("✅ Parser listo")

✅ Parser listo


In [ ]:
# Round-trip: Bell State
qc_original = QuantumCircuit(2)
qc_original.h(0)
qc_original.cx(0, 1)

recovered_dict = circuit_to_json(qc_original, level=99)
qc_restored    = parse_json_to_circuit(recovered_dict)

# Comparar sin mediciones
qc_restored_no_measure = qc_restored.copy()
qc_restored_no_measure.remove_final_measurements()

assert qc_restored_no_measure.num_qubits == qc_original.num_qubits
assert qc_restored_no_measure.count_ops() == qc_original.count_ops()

print("✅ Round-trip Bell State correcto")
print(f"   Ops original : {qc_original.count_ops()}")
print(f"   Ops restaurado: {qc_restored_no_measure.count_ops()}")

✅ Round-trip Bell State correcto
   Ops original : OrderedDict({'h': 1, 'cx': 1})
   Ops restaurado: OrderedDict({'measure': 2, 'h': 1, 'cx': 1})


## 🖼️ 8. Visualizar el circuito restaurado

In [15]:
qc_draw = qc_restored.copy()
qc_draw.remove_final_measurements()

print("🔭 Circuito Bell restaurado desde JSON:")
qc_draw.draw()

🔭 Circuito Bell restaurado desde JSON:


┌───┐ CNOT 
q_0: ┤ H ├──■───
     └───┘┌─┴─┐ 
q_1: ─────┤ X ├─
          └───┘